# **Laboratorio 8: Ready, Set, Deploy! 👩‍🚀👨‍🚀**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2026 </strong></center>

### Cuerpo Docente:

- Profesores: Pablo Badilla, Diego Cortez
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Jiale Chen
- Nombre de alumno 2:

### **Link de repositorio de GitHub:** [Repositorio](https://github.com/Fa11ingDeep/MDS7202)

## Temas a tratar

- Entrenamiento y registro de modelos usando MLFlow.
- Despliegue de modelo usando FastAPI
- Containerización del proyecto usando Docker

### Objetivos principales del laboratorio

- Generar una solución a un problema a partir de ML
- Desplegar su solución usando MLFlow, FastAPI y Docker

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# **Introducción**

<p align="center">
  <img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExODJnMHJzNzlkNmQweXoyY3ltbnZ2ZDlxY2c0aW5jcHNzeDNtOXBsdCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/AbPdhwsMgjMjax5reo/giphy.gif" width="400">
</p>



Consumida en la tristeza el despido de Renacín, Smapina ha decaído en su desempeño, lo que se ha traducido en un irregular tratamiento del agua. Esto ha implicado una baja en la calidad del agua, llegando a haber algunos puntos de la comuna en la que el vital elemento no es apto para el consumo humano. Es por esto que la sanitaria pública de la municipalidad de Maipú se ha contactado con ustedes para que le entreguen una urgente solución a este problema (a la vez que dejan a Smapina, al igual que Renacín, sin trabajo 😔).

El problema que la empresa le ha solicitado resolver es el de elaborar un sistema que les permita saber si el agua es potable o no. Para esto, la sanitaria les ha proveido una base de datos con la lectura de múltiples sensores IOT colocados en diversas cañerías, conductos y estanques. Estos sensores señalan nueve tipos de mediciones químicas y más una etiqueta elaborada en laboratorio que indica si el agua es potable o no el agua.

La idea final es que puedan, en el caso que el agua no sea potable, dar un aviso inmediato para corregir el problema. Tenga en cuenta que parte del equipo docente vive en Maipú y su intoxicación podría implicar graves problemas para el cierre del curso.

Atributos:

1. pH value
2. Hardness
3. Solids (Total dissolved solids - TDS)
4. Chloramines
5. Sulfate
6. Conductivity
7. Organic_carbon
8. Trihalomethanes
9. Turbidity

Variable a predecir:

10. Potability (1 si es potable, 0 no potable)

Descripción de cada atributo se pueden encontrar en el siguiente link: [dataset](https://www.kaggle.com/adityakadiwal/water-potability)

# **1. Optimización de modelos con Optuna + MLFlow (2.0 puntos)**

El objetivo de esta sección es que ustedes puedan combinar Optuna con MLFlow para poder realizar la optimización de los hiperparámetros de sus modelos.

Como aún no hemos hablado nada sobre `MLFlow` cabe preguntarse: **¡¿Qué !"#@ es `MLflow`?!**

<p align="center">
  <img src="https://media.tenor.com/eusgDKT4smQAAAAC/matthew-perry-chandler-bing.gif" width="400">
</p>

## **MLFlow**

`MLflow` es una plataforma de código abierto que simplifica la gestión y seguimiento de proyectos de aprendizaje automático. Con sus herramientas, los desarrolladores pueden organizar, rastrear y comparar experimentos, además de registrar modelos y controlar versiones.

<p align="center">
  <img src="https://spark.apache.org/images/mlflow-logo.png" width="350">
</p>

Si bien esta plataforma cuenta con un gran número de herramientas y funcionalidades, en este laboratorio trabajaremos con dos:
1. **Runs**: Registro que constituye la información guardada tras la ejecución de un entrenamiento. Cada `run` tiene su propio run_id, el cual sirve como identificador para el entrenamiento en sí mismo. Dentro de cada `run` podremos acceder a información como los hiperparámetros utilizados, las métricas obtenidas, las librerías requeridas y hasta nos permite descargar el modelo entrenado.
2. **Experiments**: Se utilizan para agrupar y organizar diferentes ejecuciones de modelos (`runs`). En ese sentido, un experimento puede agrupar 1 o más `runs`. De esta manera, es posible también registrar métricas, parámetros y archivos (artefactos) asociados a cada experimento.

### **Todo bien pero entonces, ¿cómo se usa en la práctica `MLflow`?**

Es sencillo! Considerando un problema de machine learning genérico, podemos registrar la información relevante del entrenamiento ejecutando `mlflow.autolog()` antes entrenar nuestro modelo. Veamos este bonito ejemplo facilitado por los mismos creadores de `MLflow`:

```python
#!pip install mlflow
import mlflow # importar mlflow

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog() # registrar automáticamente información del entrenamiento
with mlflow.start_run(): # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train) # train the model
    predictions = rf.predict(X_test) # Use the model to make predictions on the test dataset.
    # aquí termina el run
```

Si ustedes ejecutan el código anterior en sus máquinas locales (desde un jupyter notebook por ejemplo) se darán cuenta que en su directorio *root* se ha creado la carpeta `mlruns`. Esta carpeta lleva el tracking de todos los entrenamientos ejecutados desde el directorio root (importante: si se cambian de directorio y vuelven a ejecutar el código anterior, se creará otra carpeta y no tendrán acceso al entrenamiento anterior). Para visualizar estos entrenamientos, `MLflow` nos facilita hermosa interfaz visual a la que podemos acceder ejecutando:

```
mlflow ui
```

y luego pinchando en la ruta http://127.0.0.1:5000 que nos retorna la terminal. Veamos en vivo algunas de sus funcionalidades!

<p align="center">
  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZXVuM3A5MW1heDFpa21qbGlwN2pyc2VoNnZsMmRzODZxdnluemo2bCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3o84sq21TxDH6PyYms/giphy.gif" width="400">
</p>

Les dejamos también algunos comandos útiles:

- `mlflow.create_experiment("nombre_experimento")`: Les permite crear un nuevo experimento para agrupar entrenamientos
- `mlflow.log_metric("nombre_métrica", métrica)`: Les permite registrar una métrica *custom* bajo el nombre de "nombre_métrica"


In [1]:
!uv add mlflow

Resolved 190 packages in 2ms
Checked 182 packages in 26ms


Si tiene problemas puede necesitar ejecutar `uv add "setuptools<82.0.0"`

In [2]:
import mlflow  # importar mlflow
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog()  # registrar automáticamente información del entrenamiento
with mlflow.start_run():  # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train)  # train the model
    predictions = rf.predict(X_test)  # Use the model to make predictions on the test dataset.
    # aquí termina el run

2026/06/08 18:15:48 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.6.1, but the installed version is 1.7.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/06/08 18:15:48 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [3]:
run = mlflow.last_active_run()
info = mlflow.get_run(run.info.run_id)
print(info.data.params)
print(info.data.metrics)

{'bootstrap': 'True', 'ccp_alpha': '0.0', 'criterion': 'squared_error', 'max_depth': '6', 'max_features': '3', 'max_leaf_nodes': 'None', 'max_samples': 'None', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '1', 'min_samples_split': '2', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_estimators': '100', 'n_jobs': 'None', 'oob_score': 'False', 'random_state': 'None', 'verbose': '0', 'warm_start': 'False'}
{'training_mean_absolute_error': 30.04953826246247, 'training_mean_squared_error': 1283.1256377593045, 'training_r2_score': 0.7818594781299693, 'training_root_mean_squared_error': 35.82074312126012, 'training_score': 0.7818594781299693}


## **1.1 Combinando Optuna + MLflow (2.0 puntos)**

Ahora que tenemos conocimiento de ambas herramientas, intentemos ahora combinarlas para **más sabor**. El objetivo de este apartado es simple: automatizar la optimización de los parámetros de nuestros modelos usando `Optuna` y registrando de forma automática cada resultado en `MLFlow`.

Considerando el objetivo planteado, se le pide completar la función `optimize_model`, la cual debe:
- **Optimizar los hiperparámetros del modelo `XGBoost` usando `Optuna`.** Realice una cantidad de iteraciones para evitar tiempos de ejecución excesivos (al menos 10)
- **Registrar cada entrenamiento en un experimento nuevo**, asegurándose de que la métrica `f1-score` se registre como `"valid_f1"`. No se deben guardar todos los experimentos en *Default*; en su lugar, cada `experiment` y `run` deben tener nombres interpretables, reconocibles y diferentes a los nombres por defecto (por ejemplo, para un run: "XGBoost con lr 0.1").
- **Devolver el mejor modelo** usando la función `get_best_model` y serializarlo en el disco con `pickle.dump`. Luego, guardar el modelo en la carpeta `/models`.
- **Guardar el código en `optimize.py`**. La ejecución de `python optimize.py` debería ejecutar la función `optimize_model`.
- **Guardar las versiones de las librerías utilizadas** en el desarrollo.

*Hint: Le puede ser útil revisar los parámetros que recibe `mlflow.start_run`*

```python
def get_best_model(experiment_id):
    runs = mlflow.search_runs(experiment_id)
    best_model_id = runs.sort_values("metrics.valid_f1")["run_id"].iloc[0]
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")

    return best_model
```

En esta parte se puede ejecutar el código en el colab o ejecutar con python optimize.py, sin embargo para la segunda opción es necesario activar el venv y acceder a la carpeta lab_8 (por ejemplo C:\MDS7202\labs\lab_8), los resultados .db no se subirán a Git Hub, pero estarán disponibles en la entrega.

In [4]:
import pickle
from pathlib import Path

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import optuna
import pandas as pd
import sklearn
import xgboost
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Ruta del dataset
DATA_PATH = "water_potability.csv"

# Nombre del experimento en MLflow
EXPERIMENT_NAME = "optuna_xgboost_water_potability"


def get_best_model(experiment_id):
    # Obtener todos los runs del experimento
    runs = mlflow.search_runs(experiment_id)
    # Seleccionar el run con mejor F1
    best_model_id = runs.sort_values(
        "metrics.valid_f1",
        ascending=False,
    )["run_id"].iloc[0]
    # Cargar el modelo asociado a ese run
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")
    return best_model


def optimize_model():
    # Desactivar autologging para evitar runs automáticos
    mlflow.autolog(disable=True)
    mlflow.xgboost.autolog(disable=True)
    # Crear carpeta de salida
    Path("models").mkdir(exist_ok=True)
    # Cargar dataset
    df = pd.read_csv(DATA_PATH)
    # Separar variables predictoras y objetivo
    X = df.drop(columns=["Potability"])
    y = df["Potability"]
    # Dividir datos en entrenamiento y validación
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )
    # Crear o seleccionar experimento
    mlflow.set_experiment(EXPERIMENT_NAME)
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

    def objective(trial):
        # Espacio de búsqueda de hiperparámetros
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
        }
        # Nombre descriptivo del run
        run_name = f"XGBoost lr={params['learning_rate']:.3f}, depth={params['max_depth']}"
        # Iniciar run en MLflow
        with mlflow.start_run(
            experiment_id=experiment.experiment_id,
            run_name=run_name,
        ):
            # Forzar nombre y metadata del run
            mlflow.set_tag("mlflow.runName", run_name)
            mlflow.set_tag("model", "XGBoost")
            mlflow.set_tag("optimizer", "Optuna")
            # Crear modelo
            model = XGBClassifier(
                **params,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
            )
            # Entrenar modelo
            model.fit(X_train, y_train)
            # Realizar predicciones
            y_pred = model.predict(X_valid)
            # Calcular F1-score
            valid_f1 = f1_score(y_valid, y_pred)
            # Registrar parámetros y métrica
            mlflow.log_params(params)
            mlflow.log_metric("valid_f1", valid_f1)
            # Guardar modelo en MLflow
            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="model",
            )
        return valid_f1

    # Crear estudio Optuna
    study = optuna.create_study(
        direction="maximize",
        study_name="optuna_xgboost_water_potability_study",
    )
    # Ejecutar optimización
    study.optimize(objective, n_trials=10)
    # Guardar mejores parámetros
    pd.DataFrame([study.best_params]).to_csv(
        "models/best_params.csv",
        index=False,
    )
    # Recuperar mejor modelo registrado
    best_model = get_best_model(experiment.experiment_id)
    # Serializar mejor modelo
    with open("models/best_model.pkl", "wb") as file:
        pickle.dump(best_model, file)
    # Guardar versiones utilizadas
    versions = {
        "mlflow": mlflow.__version__,
        "optuna": optuna.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "xgboost": xgboost.__version__,
    }
    pd.DataFrame.from_dict(
        versions,
        orient="index",
        columns=["version"],
    ).to_csv(
        "models/library_versions.csv",
        index_label="library",
    )
    return best_model


if __name__ == "__main__":
    optimize_model()

2026/06/08 18:15:55 WARNING mlflow.utils.autologging_utils: MLflow xgboost autologging is known to be compatible with 1.4.2 <= xgboost <= 3.0.0, but the installed version is 3.2.0. If you encounter errors during autologging, try upgrading / downgrading xgboost to a compatible version, or try upgrading MLflow.
2026/06/08 18:15:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
[I 2026-06-08 18:15:55,254] A new study created in memory with name: optuna_xgboost_water_potability_study
2026/06/08 18:16:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
[I 2026-06-08 18:16:02,167] Trial 0 finished with value: 0.2857142857142857 and parameters: {'n_estimators': 84, 'max_depth': 8, 'learning_rate': 0.011022842517055044, 'subsample': 0.9205261749972771, 'colsample_bytree': 0.9693084616647604, 'gamma': 4.039520913313234, 'reg_alpha': 0.2570992

# **2. FastAPI (2.0 puntos)**

<div align="center">
  <img src="https://media3.giphy.com/media/YQitE4YNQNahy/giphy-downsized-large.gif" width="500">
</div>

Con el modelo ya entrenado, la idea de esta sección es generar una API REST a la cual se le pueda hacer *requests* para así interactuar con su modelo. En particular, se le pide:

- Guardar el código de esta sección en el archivo `main.py`. Note que ejecutar `python main.py` debería levantar el servidor en el puerto por defecto.
- Defina `GET` con ruta tipo *home* que describa brevemente su modelo, el problema que intenta resolver, su entrada y salida.
- Defina un `POST` a la ruta `/potabilidad/` donde utilice su mejor optimizado para predecir si una medición de agua es o no potable. Por ejemplo, una llamada de esta ruta con un *body*:

```json
{
   "ph":10.316400384553162,
   "Hardness":217.2668424334475,
   "Solids":10676.508475429378,
   "Chloramines":3.445514571005745,
   "Sulfate":397.7549459751925,
   "Conductivity":492.20647361771086,
   "Organic_carbon":12.812732207582542,
   "Trihalomethanes":72.28192021570328,
   "Turbidity":3.4073494284238364
}
```

Su servidor debería retornar una respuesta HTML con código 200 con:


```json
{
  "potabilidad": 0 # respuesta puede variar según el clasificador que entrenen
}
```

**`HINT:` Recuerde que puede utilizar [http://localhost:8000/docs](http://localhost:8000/docs) para hacer un `POST`.**

Para ejecutar python main.py es necesario activar el venv y acceder a la carpeta lab_8 (por ejemplo C:\MDS7202\labs\lab_8), además, es necesario tener el mejor modelo en el directorio como se menciona en la parte anterior, también es necesario agregar FastApi y uvicorn.

# **3. Docker (2 puntos)**

<div align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/1*9rafh2W0rbRJIKJzqYc8yA.gif" width="500">
</div>

Tras el éxito de su aplicación web para generar la salida, Smapina le solicita que genere un contenedor para poder ejecutarla en cualquier computador de la empresa de agua potable.

## **3.1 Creación de Container (1 punto)**

Cree un Dockerfile que use una imagen base de Python, copie los archivos del proyecto e instale las dependencias desde un `requirements.txt`. Con esto, construya y ejecute el contenedor Docker para la API configurada anteriormente. Entregue el código fuente (incluyendo `main.py`, `requirements.txt`, y `Dockerfile`) y la imagen Docker de la aplicación. Para la dockerización, asegúrese de cumplir con los siguientes puntos:

1. **Generar un archivo `.dockerignore`** que ignore carpetas y archivos innecesarios dentro del contenedor.
2. **Configurar un volumen** que permita la persistencia de los datos en una ruta local del computador.
3. **Exponer el puerto** para acceder a la ruta de la API sin tener que entrar al contenedor directamente.
4. **Incluir imágenes en el notebook** que muestren la ejecución del contenedor y los resultados obtenidos.
5. **Revisar y comentar los recursos utilizados por el contenedor**. Analice si los contenedores son livianos en términos de recursos.

## **3.2 Preguntas de Smapina (1 punto)**
Tras haber experimentado con Docker, Smapina desea profundizar más en el tema y decide realizarle las siguientes consultas:

- ¿Cómo se diferencia Docker de una máquina virtual (VM)?
- ¿Cuál es la diferencia entre usar Docker y ejecutar la aplicación directamente en el sistema local?
- ¿Cómo asegura Docker la consistencia entre diferentes entornos de desarrollo y producción?
- ¿Cómo se gestionan los volúmenes en Docker para la persistencia de datos?
- ¿Qué son Dockerfile y docker-compose.yml, y cuál es su propósito?

Para esta parte se comentó host="127.0.0.1" en main.py, dado que ahora el docker recibe solicitudes desde afuera.

para crear la imagen se utiliza el siguiente comando:
```bash
docker build -t water-potability-api .
```

Para ejecutar el docker se usa el siguiente comando:

```bash
docker run --name water-api `
  -p 8000:8000 `
  -v ${PWD}/models:/app/models `
  water-potability-api
```
Lo anterior deja disponible el puerto 8000 y setea el volumen del docker de manera dinámica considerando el tamaño del modelo.

(En el caso de que ya se ejecutó, se puede volver a inicializar y detener con los siguientes comandos, considerando que el nombre es 'water-api')

```bash
docker start water-api
```

```bash
docker stop water-api
```

Imágen de la ejecución y los resultados:

<div align="center">
    <img src="imagenes/resultados.png" width="1000">
</div>


Se monitoreó el contenedor utilizando el comando `docker stats`. Los resultados obtenidos fueron los siguientes:

| Métrica | Valor |
|----------|--------|
| CPU | 0.28 % |
| Memoria utilizada | 252.5 MiB |
| Memoria disponible | 7.623 GiB |
| Uso de memoria | 3.23 % |
| Red (entrada/salida) | 6.98 kB / 5.76 kB |
| Disco (lectura/escritura) | 0 B / 0 B |
| Procesos (PIDs) | 137 |

A partir de estos resultados, se observa que el contenedor presenta un consumo muy bajo de CPU (**0.28 %**), lo que indica que la API requiere pocos recursos computacionales para realizar inferencias con el modelo entrenado.

Respecto a la memoria, el contenedor utiliza aproximadamente **252.5 MiB**, equivalente al **3.23 %** de la memoria disponible. Este consumo es razonable considerando que se mantienen cargados en memoria el modelo XGBoost y librerías como Pandas y Scikit-Learn.

El tráfico de red registrado es mínimo, ya que la aplicación únicamente recibe solicitudes HTTP y retorna predicciones. Asimismo, no se observó actividad significativa de lectura o escritura en disco durante la ejecución.

En conclusión, el contenedor puede considerarse **liviano en términos de recursos**, especialmente en consumo de CPU. El uso de memoria es moderado y adecuado para una aplicación de Machine Learning desplegada mediante FastAPI. Además, el uso de la imagen base `python:3.12-slim` contribuye a reducir el tamaño de la imagen y los recursos requeridos por el contenedor.

1- Docker y las máquinas virtuales permiten ejecutar aplicaciones de forma aislada, pero utilizan enfoques distintos. Una máquina virtual incluye un sistema operativo completo, por lo que consume más memoria, almacenamiento y recursos de procesamiento.

En cambio, Docker utiliza contenedores que comparten el sistema operativo anfitrión y solo incluyen la aplicación y sus dependencias. Gracias a esto, los contenedores son más livianos, ocupan menos espacio y se inician más rápido.

Por estas características, Docker es una solución eficiente para desplegar aplicaciones como la API desarrollada en este laboratorio.

2-  Al ejecutar la aplicación directamente en el sistema local, es necesario instalar y configurar manualmente todas las dependencias requeridas. Esto puede generar problemas de compatibilidad entre distintos computadores o sistemas operativos.

En cambio, Docker empaqueta la aplicación junto con sus dependencias en un contenedor, garantizando que se ejecute de la misma forma en cualquier entorno donde Docker esté instalado. Esto facilita el despliegue, mejora la portabilidad y reduce errores asociados a diferencias de configuración entre equipos.

3- Docker asegura la consistencia mediante el uso de **imágenes**, las cuales contienen la aplicación junto con todas sus dependencias, configuraciones y versiones específicas de software. Estas imágenes se construyen una sola vez y luego pueden ejecutarse de forma idéntica en cualquier máquina que tenga Docker instalado.

Además, Docker utiliza archivos como `Dockerfile` para definir de manera explícita cómo debe construirse el entorno de ejecución. De esta forma, tanto el entorno de desarrollo como el de producción utilizan exactamente la misma configuración, evitando diferencias en librerías, dependencias o versiones que podrían provocar errores al desplegar la aplicación.

4- Docker gestiona la persistencia de datos mediante el montaje de **volúmenes** entre el sistema anfitrión y el contenedor. Cuando se crea un volumen, Docker establece un enlace entre una carpeta externa y una ruta dentro del contenedor, de modo que las operaciones de lectura y escritura realizadas por la aplicación se almacenan fuera del contenedor.

5- Un **Dockerfile** es un archivo de texto que contiene las instrucciones necesarias para construir una imagen Docker. En él se especifica la imagen base, las dependencias a instalar, los archivos que se deben copiar y el comando que se ejecutará al iniciar el contenedor. Su propósito es definir de forma reproducible el entorno donde se ejecutará la aplicación.

Por otro lado, **docker-compose.yml** es un archivo de configuración que permite definir y administrar múltiples contenedores de manera conjunta. Mediante este archivo es posible especificar servicios, redes, volúmenes y puertos en un solo lugar, facilitando el despliegue de aplicaciones compuestas por varios componentes. Su propósito es simplificar la ejecución y coordinación de contenedores relacionados mediante un único comando.

# Conclusión

Éxito!
<div align="center">
  <img src="https://i.pinimg.com/originals/55/f5/fd/55f5fdc9455989f8caf7fca7f93bd96a.gif" width="500">
</div>